# Disease Area x GlycoEnzOnto Pathway Enrichment Analysis

Statistically demonstrate that LINCS compounds with similar **Disease Area** or **Indication** show similar **GlycoEnzOnto pathway enrichment** results.

## Analysis Flow
1. Retrieve glycogene expression data from Snowflake
2. Calculate GlycoEnzOnto pathway enrichment
3. Retrieve disease area and indication information from Drug Repurposing Hub
4. Statistical analysis (Kruskal-Wallis, Within vs Between similarity)
5. Visualization (dot plot, violin plot)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import logging
import requests
import time
import json
from typing import Dict, List, Optional, Tuple, Set
from scipy.stats import fisher_exact, kruskal, mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

from utils.pathway_abbreviations import abbreviate_pathway

# Figure style settings
plt.rcParams.update({
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'font.size': 16,
    'svg.fonttype': 'none',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'lines.linewidth': 1.5,
})

# Results directory
results_dir = project_root / 'results' / 'atc_glycopathway'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Results directory: {results_dir}')

## 1. Snowflake Connection Settings

In [ ]:
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect

def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH", "~/.ssh/snowflake_rsa_key.pem")')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user=os.environ.get("SNOWFLAKE_USER"),
            account=os.environ.get("SNOWFLAKE_ACCOUNT"),
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

print("Snowflake connection functions defined")

## 2. Data Retrieval

In [ ]:
# 肝臓がん細胞株リストを読み込み
liver_cells_path = project_root / "results" / "liver_cell_lines_list.csv"
df_liver_list = pd.read_csv(liver_cells_path)
cell_lines = df_liver_list['CELL_LINE_NAME'].tolist()

print(f"肝臓がん細胞株リストを読み込みました: {len(cell_lines)}種類")
print(f"細胞株: {', '.join(cell_lines)}")

# Snowflakeに接続
conn = connect_to_snowflake()

In [ ]:
# glycogeneカラム名を取得
metadata_columns = {
    'VALUE', 'canonical_smiles', 'cell', 'cmapid', 'compound_alias', 
    'dose', 'inchi_key', 'pertid', 'pertname', 'timepoint', 'sample_id'
}

column_query = """
SELECT COLUMN_NAME 
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_SCHEMA = 'LINCS' 
AND TABLE_NAME = 'GLYCO_GENES_WIDE' 
ORDER BY COLUMN_NAME
"""

column_df = pd.read_sql(column_query, conn)
all_columns = column_df['COLUMN_NAME'].tolist()
glyco_genes = [col for col in all_columns if col not in metadata_columns]

print(f"糖鎖関連遺伝子カラム数: {len(glyco_genes)}")
print(f"遺伝子例: {', '.join(glyco_genes[:10])}")

In [ ]:
# 肝臓がん細胞株データを取得
gene_columns = ', '.join([f'"{gene}"' for gene in glyco_genes])
cell_lines_str = "', '".join(cell_lines)

query = f"""
SELECT 
    "sample_id",
    "canonical_smiles",
    "inchi_key",
    "pertname",
    "pertid",
    "cell",
    "dose",
    "timepoint",
    {gene_columns}
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE
WHERE "cell" IN ('{cell_lines_str}')
"""

print("クエリを実行中...")
df = pd.read_sql(query, conn)
print(f"データ取得完了: {len(df):,}レコード")
print(f"化合物数: {df['pertname'].nunique()}")
print(f"細胞株: {df['cell'].unique()}")

# InChIKeyあり化合物のフィルタリング
df_with_inchi = df[df['inchi_key'].notna() & (df['inchi_key'] != '')].copy()
print(f"\nInChIKeyあり化合物データ: {len(df_with_inchi):,}レコード")
print(f"InChIKeyあり化合物数: {df_with_inchi['pertname'].nunique()}")

# 数値型に変換
for gene in glyco_genes:
    if gene in df_with_inchi.columns:
        df_with_inchi[gene] = pd.to_numeric(df_with_inchi[gene], errors='coerce')

df_with_inchi.head()

## 3. Loading GlycoEnzOnto Pathway Definitions

In [ ]:
# GlycoEnzOnto GMTファイルを読み込み
gmt_file = project_root / 'GlycoEnzOnto' / 'GlycoEnzOnto.gmt'

def read_gmt(gmt_path):
    """GMTファイルを読み込んでパスウェイ辞書を返す"""
    pathways = {}
    descriptions = {}
    with open(gmt_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0].strip('"').strip("'")
            description = parts[1].strip('"').strip("'") if len(parts) > 1 else ''
            genes = [g.strip('"').strip("'") for g in parts[2:]]
            pathways[pathway_name] = set(genes)
            descriptions[pathway_name] = description
    return pathways, descriptions

pathways, pathway_descriptions = read_gmt(gmt_file)

print(f'GlycoEnzOnto パスウェイ情報:')
print(f'  総パスウェイ数: {len(pathways)}')

# パスウェイサイズの分布
pathway_sizes = [len(genes) for genes in pathways.values()]
print(f'  パスウェイサイズ: 最小={min(pathway_sizes)}, 最大={max(pathway_sizes)}, 平均={np.mean(pathway_sizes):.1f}')

print(f'\nパスウェイ例:')
for i, (name, genes) in enumerate(list(pathways.items())[:5], 1):
    print(f'    {i}. {name}: {len(genes)}遺伝子')

## 4. GSEA Function Definition

Aggregate gene expression profiles per Disease Area and perform GSEA (prerank) on GlycoEnzOnto pathways.

In [ ]:
from scipy.stats import rankdata
from statsmodels.stats.multitest import multipletests

def calculate_gene_scores(df_group: pd.DataFrame, glyco_genes: List[str]) -> pd.DataFrame:
    """
    グループのデータから各遺伝子のスコア（平均z-score）を計算
    """
    results = []
    for gene in glyco_genes:
        if gene not in df_group.columns:
            continue
        values = df_group[gene].dropna().values
        if len(values) < 3:
            continue
        mean_zscore = np.mean(values)
        std_zscore = np.std(values)
        results.append({
            'gene': gene,
            'mean_zscore': mean_zscore,
            'std_zscore': std_zscore,
            'n_samples': len(values)
        })
    
    if not results:
        return pd.DataFrame()
    
    df_scores = pd.DataFrame(results)
    df_scores = df_scores.sort_values('mean_zscore', ascending=False)
    df_scores['rank'] = range(1, len(df_scores) + 1)
    return df_scores


def calculate_enrichment_score(scores: np.ndarray, hit_indices: List[int], n_genes: int):
    """
    Enrichment Scoreを計算（GSEA標準アルゴリズム）
    """
    n_hit = len(hit_indices)
    n_miss = n_genes - n_hit
    
    hit_scores = np.abs(scores[hit_indices])
    hit_sum = np.sum(hit_scores)
    
    running_sum = 0
    max_es = 0
    min_es = 0
    max_idx = 0
    
    hit_set = set(hit_indices)
    
    for i in range(n_genes):
        if i in hit_set:
            running_sum += np.abs(scores[i]) / hit_sum if hit_sum > 0 else 1/n_hit
        else:
            running_sum -= 1 / n_miss if n_miss > 0 else 0
        
        if running_sum > max_es:
            max_es = running_sum
            max_idx = i
        if running_sum < min_es:
            min_es = running_sum
    
    if abs(max_es) >= abs(min_es):
        es = max_es
        leading_edge = [idx for idx in hit_indices if idx <= max_idx]
    else:
        es = min_es
        leading_edge = [idx for idx in hit_indices if idx >= max_idx]
    
    return es, leading_edge


def run_gsea_prerank(gene_scores: pd.DataFrame, pathways: Dict[str, Set[str]], 
                     n_permutations: int = 1000, min_size: int = 3, max_size: int = 500,
                     seed: int = 42) -> pd.DataFrame:
    """
    ランクベースGSEA（prerank版）
    """
    np.random.seed(seed)
    
    ranked_genes = gene_scores['gene'].tolist()
    scores = gene_scores['mean_zscore'].values
    n_genes = len(ranked_genes)
    gene_to_idx = {g: i for i, g in enumerate(ranked_genes)}
    
    results = []
    
    for pathway_name, pathway_genes in pathways.items():
        pathway_genes_in_list = [g for g in pathway_genes if g in gene_to_idx]
        n_hit = len(pathway_genes_in_list)
        
        if n_hit < min_size or n_hit > max_size:
            continue
        
        hit_indices = sorted([gene_to_idx[g] for g in pathway_genes_in_list])
        es, leading_edge = calculate_enrichment_score(scores, hit_indices, n_genes)
        
        null_es = []
        for _ in range(n_permutations):
            perm_indices = sorted(np.random.choice(n_genes, n_hit, replace=False))
            perm_es, _ = calculate_enrichment_score(scores, perm_indices, n_genes)
            null_es.append(perm_es)
        
        null_es = np.array(null_es)
        
        if es >= 0:
            p_value = np.mean(null_es >= es)
        else:
            p_value = np.mean(null_es <= es)
        
        if es >= 0:
            pos_null = null_es[null_es >= 0]
            nes = es / np.mean(pos_null) if len(pos_null) > 0 and np.mean(pos_null) != 0 else 0
        else:
            neg_null = null_es[null_es < 0]
            nes = es / np.abs(np.mean(neg_null)) if len(neg_null) > 0 and np.mean(neg_null) != 0 else 0
        
        le_genes = [ranked_genes[i] for i in leading_edge]
        
        results.append({
            'pathway': pathway_name,
            'es': es,
            'nes': nes,
            'p_value': max(p_value, 1/n_permutations),
            'n_genes': n_hit,
            'leading_edge_size': len(le_genes),
            'leading_edge_genes': ', '.join(le_genes[:10])
        })
    
    if not results:
        return pd.DataFrame()
    
    df_results = pd.DataFrame(results)
    _, pvals_corrected, _, _ = multipletests(df_results['p_value'], method='fdr_bh')
    df_results['q_value'] = pvals_corrected
    
    return df_results.sort_values('p_value')


print('GSEA関数を定義しました')

In [ ]:
# Drug Repurposing Hub からデータ取得
print('='*80)
print('Drug Repurposing Hub からの疾患領域情報取得')
print('='*80)

drh_query = """
SELECT 
    PERT_INAME,
    CLINICAL_PHASE,
    MOA,
    TARGET,
    DISEASE_AREA,
    INDICATION
FROM BIOINFORMATICS.LINCS.DRUG_REPURPOSING_HUB
WHERE DISEASE_AREA IS NOT NULL
"""

df_drh = pd.read_sql(drh_query, conn)
print(f'Drug Repurposing Hub データ取得: {len(df_drh):,}件')

# 化合物名でマッチング（小文字化）
df_drh['pert_iname_lower'] = df_drh['PERT_INAME'].str.lower()
df_with_inchi['pertname_lower'] = df_with_inchi['pertname'].str.lower()

# マージ
df_with_disease = df_with_inchi.merge(
    df_drh[['pert_iname_lower', 'DISEASE_AREA', 'INDICATION', 'MOA']].drop_duplicates(),
    left_on='pertname_lower',
    right_on='pert_iname_lower',
    how='inner'
)

print(f'Disease Areaありレコード数: {len(df_with_disease):,}')
print(f'Disease Areaあり化合物数: {df_with_disease["pertname"].nunique()}')

# disease_area の分布
disease_counts = df_with_disease.groupby('DISEASE_AREA')['pertname'].nunique().sort_values(ascending=False)
print(f'\nDisease Area数: {len(disease_counts)}')
print(f'\n上位15 Disease Area:')
for i, (disease, count) in enumerate(disease_counts.head(15).items(), 1):
    print(f'  {i:2d}. {disease}: {count}化合物')

In [ ]:
# Disease AreaごとにGSEA実行
print('='*80)
print('Disease AreaごとのGSEA実行')
print('='*80)

# 最小化合物数でフィルタリング
MIN_COMPOUNDS = 3
MIN_SAMPLES = 10
N_PERMUTATIONS = 1000

valid_disease_areas = disease_counts[disease_counts >= MIN_COMPOUNDS].index.tolist()
# 空白のdisease_areaを除外
valid_disease_areas = [d for d in valid_disease_areas if d and d.strip()]
print(f'有効なDisease Area数 (化合物数>={MIN_COMPOUNDS}): {len(valid_disease_areas)}')

all_gsea_results = []

for disease_area in valid_disease_areas:
    df_disease = df_with_disease[df_with_disease['DISEASE_AREA'] == disease_area]
    n_compounds = df_disease['pertname'].nunique()
    n_samples = len(df_disease)
    
    if n_samples < MIN_SAMPLES:
        continue
    
    # 遺伝子スコア計算
    df_scores = calculate_gene_scores(df_disease, glyco_genes)
    
    if len(df_scores) < 50:
        continue
    
    # GSEA実行
    df_gsea = run_gsea_prerank(df_scores, pathways, n_permutations=N_PERMUTATIONS, min_size=3)
    
    if len(df_gsea) == 0:
        continue
    
    # 有意な結果のみ
    df_sig = df_gsea[df_gsea['q_value'] < 0.25].copy()
    
    n_sig = (df_gsea['q_value'] < 0.05).sum()
    n_sig_25 = len(df_sig)
    
    if n_sig_25 > 0:
        print(f'{disease_area}: {n_compounds}化合物, {n_samples}サンプル → 有意パスウェイ(q<0.05): {n_sig}, (q<0.25): {n_sig_25}')
        df_sig['disease_area'] = disease_area
        df_sig['n_compounds'] = n_compounds
        df_sig['n_samples'] = n_samples
        all_gsea_results.append(df_sig)

# 結果統合
if all_gsea_results:
    df_gsea_all = pd.concat(all_gsea_results, ignore_index=True)
    print(f'\n総GSEA結果: {len(df_gsea_all)}レコード')
    print(f'有意なDisease Area数: {df_gsea_all["disease_area"].nunique()}')
    
    # 保存
    df_gsea_all.to_csv(results_dir / 'disease_area_pathway_gsea.csv', index=False)
    print(f'保存しました: {results_dir / "disease_area_pathway_gsea.csv"}')
    
    # サマリー表示
    print(f'\n【頻出パスウェイ Top10】')
    pathway_counts = df_gsea_all.groupby('pathway').size().sort_values(ascending=False)
    for pathway, count in pathway_counts.head(10).items():
        print(f'  {pathway}: {count} Disease Area')
else:
    df_gsea_all = pd.DataFrame()
    print('有意なGSEA結果なし')

## 5. Visualization

In [ ]:
# Disease Area × Pathway ドットプロット
print('='*80)
print('ドットプロット: Disease Area × Pathway (GSEA)')
print('='*80)

if len(df_gsea_all) > 0:
    # ピボットテーブル作成（NESを使用）
    df_pivot = df_gsea_all.pivot_table(
        index='disease_area',
        columns='pathway',
        values='nes',
        aggfunc='mean'
    ).fillna(0)
    
    # 上位パスウェイ選択（分散が大きいもの）
    pathway_var = df_pivot.var().sort_values(ascending=False)
    top_pathways = pathway_var.head(25).index.tolist()
    
    # 上位Disease Area選択（サンプル数が多いもの）
    disease_sample_counts = df_gsea_all.groupby('disease_area')['n_samples'].first().sort_values(ascending=False)
    top_diseases = disease_sample_counts.head(15).index.tolist()
    
    df_plot = df_pivot.loc[
        [d for d in top_diseases if d in df_pivot.index],
        [p for p in top_pathways if p in df_pivot.columns]
    ]
    
    print(f'プロットサイズ: {df_plot.shape}')
    
    if df_plot.shape[0] >= 2 and df_plot.shape[1] >= 2:
        fig, ax = plt.subplots(figsize=(14, 10))
        
        plot_data = []
        for disease in df_plot.index:
            for pathway in df_plot.columns:
                nes = df_plot.loc[disease, pathway]
                if nes != 0:
                    mask = (df_gsea_all['disease_area'] == disease) & (df_gsea_all['pathway'] == pathway)
                    if mask.any():
                        row = df_gsea_all[mask].iloc[0]
                        plot_data.append({
                            'disease_area': disease,
                            'pathway': abbreviate_pathway(pathway),
                            'nes': nes,
                            'n_genes': row['n_genes'],
                            'q_value': row['q_value']
                        })
        
        df_plot_data = pd.DataFrame(plot_data)
        
        if len(df_plot_data) > 0:
            max_abs_nes = df_plot_data['nes'].abs().max()
            
            scatter = ax.scatter(
                df_plot_data['disease_area'],
                df_plot_data['pathway'],
                c=df_plot_data['nes'],
                s=df_plot_data['n_genes'] * 10 + 20,
                cmap='RdBu_r',
                vmin=-max_abs_nes,
                vmax=max_abs_nes,
                edgecolors='black',
                linewidth=0.5,
                alpha=0.8
            )
            
            plt.colorbar(scatter, ax=ax, shrink=0.6, label='NES\n(red=up, blue=down)')
            
            ax.set_xlabel('Disease Area', fontweight='bold')
            ax.set_ylabel('GlycoEnzOnto', fontweight='bold')
            ax.set_title('Disease Area × GlycoEnzOnto Pathway GSEA', fontweight='bold')
            
            plt.xticks(rotation=45, ha='right')
            plt.yticks(fontsize=10)
            ax.grid(True, alpha=0.3, linestyle='--')
            
            plt.tight_layout()
            
            fig.savefig(results_dir / 'fig_disease_area_gsea_dotplot.png', dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(results_dir / 'fig_disease_area_gsea_dotplot.pdf', bbox_inches='tight', facecolor='white')
            fig.savefig(results_dir / 'fig_disease_area_gsea_dotplot.svg', format='svg', bbox_inches='tight', facecolor='white')
            print(f'保存しました: {results_dir / "fig_disease_area_gsea_dotplot.png"}')
            
            plt.show()
else:
    print('GSEA結果がありません')

In [ ]:
# Disease Area間 類似度比較（NESベース）- バイオリンプロット
print('='*80)
print('Disease Area間 類似度分析')
print('='*80)

if len(df_gsea_all) > 0:
    # Disease Area × Pathway マトリックス
    df_nes_matrix = df_gsea_all.pivot_table(
        index='disease_area',
        columns='pathway',
        values='nes',
        aggfunc='mean',
        fill_value=0
    )
    
    from sklearn.metrics.pairwise import cosine_similarity
    
    X = df_nes_matrix.values
    disease_areas = df_nes_matrix.index.tolist()
    
    cos_sim = cosine_similarity(X)
    
    # ペアワイズ類似度を収集
    pairwise_sims = []
    for i in range(len(disease_areas)):
        for j in range(i+1, len(disease_areas)):
            pairwise_sims.append({
                'disease_area_1': disease_areas[i],
                'disease_area_2': disease_areas[j],
                'similarity': cos_sim[i, j]
            })
    
    df_sims = pd.DataFrame(pairwise_sims)
    
    print(f'Disease Areaペア数: {len(df_sims):,}')
    print(f'平均類似度: {df_sims["similarity"].mean():.4f} (SD: {df_sims["similarity"].std():.4f})')
    
    # バイオリンプロット
    fig, ax = plt.subplots(figsize=(8, 6))
    
    parts = ax.violinplot([df_sims['similarity'].values], positions=[0], showmeans=True, showmedians=True)
    
    for pc in parts['bodies']:
        pc.set_facecolor('steelblue')
        pc.set_alpha(0.7)
    
    # 統計情報を追加
    mean_sim = df_sims['similarity'].mean()
    std_sim = df_sims['similarity'].std()
    
    ax.axhline(mean_sim, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_sim:.3f}')
    ax.text(0.3, mean_sim, f'Mean: {mean_sim:.3f}\nSD: {std_sim:.3f}',
            verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_ylabel('Cosine Similarity (NES profile)', fontweight='bold')
    ax.set_title('Disease Area Pairwise Similarity Distribution', fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels(['Between Disease Areas'])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    
    fig.savefig(results_dir / 'fig_disease_area_similarity_violin.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(results_dir / 'fig_disease_area_similarity_violin.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(results_dir / 'fig_disease_area_similarity_violin.svg', format='svg', bbox_inches='tight', facecolor='white')
    print(f'保存しました: {results_dir / "fig_disease_area_similarity_violin.png"}')
    plt.show()
else:
    print('GSEA結果がありません')

## 6. Indication-based GSEA Analysis

In [ ]:
# Indication（適応症）によるGSEA
print('='*80)
print('IndicationごとのGSEA実行')
print('='*80)

# Indication分布
indication_counts = df_with_disease.groupby('INDICATION')['pertname'].nunique().sort_values(ascending=False)
# 空白を除外
indication_counts = indication_counts[indication_counts.index.str.strip() != '']

print(f'Indication数: {len(indication_counts)}')
print(f'\n上位15 Indication:')
for i, (ind, count) in enumerate(indication_counts.head(15).items(), 1):
    print(f'  {i:2d}. {ind}: {count}化合物')

# GSEAを実行
valid_indications = indication_counts[indication_counts >= 3].index.tolist()
print(f'\n有効なIndication数 (化合物数>=3): {len(valid_indications)}')

all_indication_gsea = []

for indication in valid_indications[:20]:  # 上位20に限定
    df_ind = df_with_disease[df_with_disease['INDICATION'] == indication]
    n_compounds = df_ind['pertname'].nunique()
    n_samples = len(df_ind)
    
    if n_samples < 10:
        continue
    
    df_scores = calculate_gene_scores(df_ind, glyco_genes)
    
    if len(df_scores) < 50:
        continue
    
    df_gsea = run_gsea_prerank(df_scores, pathways, n_permutations=500, min_size=3)
    
    if len(df_gsea) == 0:
        continue
    
    df_sig = df_gsea[df_gsea['q_value'] < 0.25].copy()
    
    if len(df_sig) > 0:
        n_sig_05 = (df_gsea['q_value'] < 0.05).sum()
        print(f'{indication[:50]}...: 有意パスウェイ(q<0.05): {n_sig_05}')
        df_sig['indication'] = indication
        df_sig['n_compounds'] = n_compounds
        df_sig['n_samples'] = n_samples
        all_indication_gsea.append(df_sig)

if all_indication_gsea:
    df_indication_gsea = pd.concat(all_indication_gsea, ignore_index=True)
    print(f'\nIndication GSEA結果: {len(df_indication_gsea)}レコード')
    df_indication_gsea.to_csv(results_dir / 'indication_pathway_gsea.csv', index=False)
    print(f'保存しました: {results_dir / "indication_pathway_gsea.csv"}')
else:
    df_indication_gsea = pd.DataFrame()
    print('有意なIndication GSEA結果なし')


In [ ]:
# Indication × Pathway ドットプロット
print('='*80)
print('ドットプロット: Indication × Pathway (GSEA)')
print('='*80)

if 'df_indication_gsea' in dir() and len(df_indication_gsea) > 0:
    # ピボットテーブル作成（NESを使用）
    df_pivot_ind = df_indication_gsea.pivot_table(
        index='indication',
        columns='pathway',
        values='nes',
        aggfunc='mean'
    ).fillna(0)
    
    # 上位パスウェイ選択（分散が大きいもの）
    pathway_var_ind = df_pivot_ind.var().sort_values(ascending=False)
    top_pathways_ind = pathway_var_ind.head(20).index.tolist()
    
    # 上位Indication選択（サンプル数が多いもの）
    ind_sample_counts = df_indication_gsea.groupby('indication')['n_samples'].first().sort_values(ascending=False)
    top_indications = ind_sample_counts.head(15).index.tolist()
    
    df_plot_ind = df_pivot_ind.loc[
        [d for d in top_indications if d in df_pivot_ind.index],
        [p for p in top_pathways_ind if p in df_pivot_ind.columns]
    ]
    
    print(f'プロットサイズ: {df_plot_ind.shape}')
    
    if df_plot_ind.shape[0] >= 2 and df_plot_ind.shape[1] >= 2:
        fig, ax = plt.subplots(figsize=(14, 10))
        
        plot_data = []
        for indication in df_plot_ind.index:
            for pathway in df_plot_ind.columns:
                nes = df_plot_ind.loc[indication, pathway]
                if nes != 0:
                    mask = (df_indication_gsea['indication'] == indication) & (df_indication_gsea['pathway'] == pathway)
                    if mask.any():
                        row = df_indication_gsea[mask].iloc[0]
                        plot_data.append({
                            'indication': indication[:40] + '...' if len(indication) > 40 else indication,
                            'pathway': abbreviate_pathway(pathway),
                            'nes': nes,
                            'n_genes': row['n_genes'],
                            'q_value': row['q_value']
                        })
        
        df_plot_data = pd.DataFrame(plot_data)
        
        if len(df_plot_data) > 0:
            max_abs_nes = df_plot_data['nes'].abs().max()
            
            scatter = ax.scatter(
                df_plot_data['indication'],
                df_plot_data['pathway'],
                c=df_plot_data['nes'],
                s=df_plot_data['n_genes'] * 10 + 20,
                cmap='RdBu_r',
                vmin=-max_abs_nes,
                vmax=max_abs_nes,
                edgecolors='black',
                linewidth=0.5,
                alpha=0.8
            )
            
            plt.colorbar(scatter, ax=ax, shrink=0.6, label='NES\n(red=up, blue=down)')
            
            ax.set_xlabel('Indication', fontweight='bold')
            ax.set_ylabel('GlycoEnzOnto', fontweight='bold')
            ax.set_title('Indication × GlycoEnzOnto Pathway GSEA', fontweight='bold')
            
            plt.xticks(rotation=45, ha='right')
            plt.yticks(fontsize=10)
            ax.grid(True, alpha=0.3, linestyle='--')
            
            plt.tight_layout()
            
            fig.savefig(results_dir / 'fig_indication_gsea_dotplot.png', dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(results_dir / 'fig_indication_gsea_dotplot.pdf', bbox_inches='tight', facecolor='white')
            fig.savefig(results_dir / 'fig_indication_gsea_dotplot.svg', format='svg', bbox_inches='tight', facecolor='white')
            print(f'保存しました: {results_dir / "fig_indication_gsea_dotplot.png"}')
            
            plt.show()
        else:
            print('プロットデータがありません')
    else:
        print(f'データが不十分です: {df_plot_ind.shape}')
else:
    print('Indication GSEA結果がありません')

## 7. Report Generation

In [ ]:
# レポート生成
print('='*80)
print('レポート生成')
print('='*80)

total_compounds = df_with_inchi['pertname'].nunique()
compounds_with_disease = df_with_disease['pertname'].nunique() if 'df_with_disease' in dir() else 0
n_disease_areas = len(valid_disease_areas) if 'valid_disease_areas' in dir() else 0
n_gsea_results = len(df_gsea_all) if 'df_gsea_all' in dir() and len(df_gsea_all) > 0 else 0
n_sig_05 = (df_gsea_all['q_value'] < 0.05).sum() if n_gsea_results > 0 else 0
n_indication_results = len(df_indication_gsea) if 'df_indication_gsea' in dir() and len(df_indication_gsea) > 0 else 0

report = f"""# Drug Repurposing Hub × GlycoEnzOnto Pathway GSEA Report

## 1. データ概要

- 総化合物数（InChIKeyあり）: {total_compounds:,}
- Disease Area情報あり化合物数: {compounds_with_disease:,} ({compounds_with_disease/total_compounds*100:.1f}%)
- 解析対象Disease Area数: {n_disease_areas}

## 2. GSEA手法

- **手法**: prerank GSEA
- **スコア**: 各グループ内の化合物サンプルから計算した遺伝子平均z-score
- **並べ替え回数**: {N_PERMUTATIONS if 'N_PERMUTATIONS' in dir() else 1000}
- **最小パスウェイサイズ**: 3
- **有意性閾値**: q-value < 0.05 (FDR補正)

## 3. Disease Area GSEA結果サマリー

- 総GSEA結果数 (q<0.25): {n_gsea_results}
- 有意なエンリッチメント (q<0.05): {n_sig_05}
- 有意なDisease Area数: {df_gsea_all['disease_area'].nunique() if n_gsea_results > 0 else 0}

## 4. Indication GSEA結果サマリー

- 総GSEA結果数 (q<0.25): {n_indication_results}
- 有意なIndication数: {df_indication_gsea['indication'].nunique() if n_indication_results > 0 else 0}

## 5. 出力ファイル

- `disease_area_pathway_gsea.csv`: Disease AreaごとのGSEA結果
- `indication_pathway_gsea.csv`: IndicationごとのGSEA結果
- `fig_disease_area_gsea_dotplot.png`: Disease Area×パスウェイドットプロット
- `fig_indication_gsea_dotplot.png`: Indication×パスウェイドットプロット
- `fig_disease_area_similarity_violin.png`: Disease Area間類似度バイオリンプロット

## 6. 図の読み方

### ドットプロット
- X軸: Disease Area / Indication
- Y軸: GlycoEnzOntoパスウェイ
- 色: NES (Normalized Enrichment Score) - 赤=上方制御、青=下方制御
- サイズ: パスウェイ内遺伝子数

### バイオリンプロット
- Disease Area間のNESプロファイルに基づくコサイン類似度の分布
"""

with open(results_dir / 'report_gsea.md', 'w') as f:
    f.write(report)

print(report)
print(f'保存しました: {results_dir / "report_gsea.md"}')


In [ ]:
# Close Snowflake connection
conn.close()
print('Snowflake connection closed')

print('\n' + '='*80)
print('All processing completed!')
print('='*80)